In [19]:
import pandas as pd

df = pd.read_csv(r'C:\Users\abhin\Dropbox\PC\Downloads\projects\Customer Churn Prediction Platform\data\Telco-Customer-Churn.csv')

In [20]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)

In [21]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')

In [22]:
df = df.drop("customerID", axis=1)

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

df_encoded = pd.get_dummies(df, drop_first=True)

In [23]:
df_encoded

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,False,True,False,False,True,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,True,False,False,True,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,True,False,False,False,True,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,1,False,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0,24,84.80,1990.50,0,True,True,True,True,False,...,False,True,False,True,True,False,True,False,False,True
7039,0,72,103.20,7362.90,0,False,True,True,True,False,...,False,True,False,True,True,False,True,True,False,False
7040,0,11,29.60,346.45,0,False,True,True,False,True,...,False,False,False,False,False,False,True,False,True,False
7041,1,4,74.40,306.60,1,True,True,False,True,False,...,False,False,False,False,False,False,True,False,False,True


## Train - Test Split

In [24]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score

X = df_encoded.drop("Churn", axis=1)
y = df_encoded["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

## Train - Model

In [25]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline

pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state = 42, max_iter=1000)
)

param_grid = {
    "logisticregression__C" : [0.01, 0.1, 1, 10],
    "logisticregression__penalty" : ['l1','l2'],
    "logisticregression__solver" :['liblinear', 'saga']
}

cv = StratifiedKFold(
    n_splits = 5,
    shuffle=True,
    random_state= 42
)

grid = GridSearchCV(
    estimator = pipeline,
    param_grid= param_grid,
    scoring='f1',
    n_jobs=-1,
    return_train_score=True,
    verbose=True,
    cv = cv
)

grid.fit(X_train, y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


c:\Users\abhin\Dropbox\PC\Downloads\projects\Customer Churn Prediction Platform\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\abhin\Dropbox\PC\Downloads\projects\Customer Churn Prediction Platform\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'logisticregression__C': [0.01, 0.1, ...], 'logisticregression__penalty': ['l1', 'l2'], 'logisticregression__solver': ['liblinear', 'saga']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",True
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``bes

In [26]:
print("Best parameters:")
print(grid.best_params_)

print("Best F1-score:")
print(grid.best_score_)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print("\n Classification Report:")
print(classification_report(y_test, y_pred))

Best parameters:
{'logisticregression__C': 10, 'logisticregression__penalty': 'l1', 'logisticregression__solver': 'liblinear'}
Best F1-score:
0.5970714518785873

 Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.65      0.56      0.60       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.74      1409
weighted avg       0.80      0.80      0.80      1409



In [27]:
import pandas as pd

results = pd.DataFrame(grid.cv_results_)

results = results[
    [
        "params",
        "mean_train_score",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values("rank_test_score")

print(results)

                                               params  mean_train_score  \
12  {'logisticregression__C': 10, 'logisticregress...          0.602634   
13  {'logisticregression__C': 10, 'logisticregress...          0.602818   
14  {'logisticregression__C': 10, 'logisticregress...          0.603056   
15  {'logisticregression__C': 10, 'logisticregress...          0.603056   
2   {'logisticregression__C': 0.01, 'logisticregre...          0.603158   
10  {'logisticregression__C': 1, 'logisticregressi...          0.602806   
8   {'logisticregression__C': 1, 'logisticregressi...          0.602110   
9   {'logisticregression__C': 1, 'logisticregressi...          0.602257   
11  {'logisticregression__C': 1, 'logisticregressi...          0.602408   
6   {'logisticregression__C': 0.1, 'logisticregres...          0.600338   
7   {'logisticregression__C': 0.1, 'logisticregres...          0.599067   
5   {'logisticregression__C': 0.1, 'logisticregres...          0.596217   
4   {'logisticregression_

### Randomized Search - Logistic Regression

In [28]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline

pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state = 42, max_iter=1000)
)

param_grid = {
    "logisticregression__C" : [0.01, 0.1, 1, 10],
    "logisticregression__penalty" : ['l1','l2'],
    "logisticregression__solver" :['liblinear', 'saga']
}

cv = StratifiedKFold(
    n_splits = 5,
    shuffle=True,
    random_state= 42
)

grid = RandomizedSearchCV(
    estimator = pipeline,
    param_distributions = param_grid,
    scoring='f1',
    n_jobs=-1,
    return_train_score=True,
    verbose=True,
    cv = cv
)

grid.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


c:\Users\abhin\Dropbox\PC\Downloads\projects\Customer Churn Prediction Platform\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\abhin\Dropbox\PC\Downloads\projects\Customer Churn Prediction Platform\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'logisticregression__C': [0.01, 0.1, ...], 'logisticregression__penalty': ['l1', 'l2'], 'logisticregression__solver': ['liblinear', 'saga']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",True
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metr

In [29]:
print("Best parameters:")
print(grid.best_params_)

print("Best F1-score:")
print(grid.best_score_)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print("\n Classification Report:")
print(classification_report(y_test, y_pred))

Best parameters:
{'logisticregression__solver': 'liblinear', 'logisticregression__penalty': 'l1', 'logisticregression__C': 10}
Best F1-score:
0.5970714518785873

 Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.65      0.56      0.60       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.74      1409
weighted avg       0.80      0.80      0.80      1409



In [30]:
import pandas as pd

results = pd.DataFrame(grid.cv_results_)

results = results[
    [
        "params",
        "mean_train_score",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values("rank_test_score")

print(results)

                                              params  mean_train_score  \
1  {'logisticregression__solver': 'liblinear', 'l...          0.602634   
9  {'logisticregression__solver': 'saga', 'logist...          0.602818   
0  {'logisticregression__solver': 'liblinear', 'l...          0.603056   
8  {'logisticregression__solver': 'liblinear', 'l...          0.603158   
6  {'logisticregression__solver': 'saga', 'logist...          0.602257   
7  {'logisticregression__solver': 'liblinear', 'l...          0.600338   
3  {'logisticregression__solver': 'saga', 'logist...          0.596217   
5  {'logisticregression__solver': 'saga', 'logist...          0.589168   
2  {'logisticregression__solver': 'liblinear', 'l...          0.550123   
4  {'logisticregression__solver': 'saga', 'logist...          0.528771   

   mean_test_score  std_test_score  rank_test_score  
1         0.597071        0.029485                1  
9         0.596853        0.029541                2  
0         0.596845     

### Random Forest

In [31]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import pandas as pd

rf = RandomForestClassifier(random_state=42)

param_grid = {
    "n_estimators" : [100, 200, 300],
    "max_depth" : [None, 10, 20, 30],
    "min_samples_split" : [2, 5, 10],
    "min_samples_leaf" : [1, 2, 4],
    "max_features" : ["sqrt", "log2"]
}

cv = StratifiedKFold(
    n_splits = 5,
    shuffle= True,
    random_state=42
)

grid = GridSearchCV(
    estimator = rf,
    param_grid = param_grid,
    scoring = 'f1',
    cv= cv,
    n_jobs = -1,
    verbose=True,
    return_train_score=True
)

grid.fit(X_train, y_train)

print("Best parameters:")
print(grid.best_params_)

print("\nBest Cross Validation F1 Score:")
print(grid.best_score_)

# Best Model
best_model = grid.best_estimator_

# Predictions
y_pred = best_model.predict(X_test)

# Evaluation
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Grid Search Results
results = pd.DataFrame(grid.cv_results_)

results = results[
    [
        "params",
        "mean_train_score",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values("rank_test_score")


Fitting 5 folds for each of 216 candidates, totalling 1080 fits


KeyboardInterrupt: 

### Randomized Search Random Forest

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import pandas as pd

rf = RandomForestClassifier(random_state=42)

# Hyperparameter Grid
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}


cv = StratifiedKFold(
    n_splits = 5,
    shuffle= True,
    random_state=42
)

grid = RandomizedSearchCV(
    estimator = rf,
    param_distributions = param_grid,
    scoring = 'f1',
    cv= cv,
    n_jobs = -1,
    verbose=True,
    return_train_score=True
)

grid.fit(X_train, y_train)

print("Best parameters:")
print(grid.best_params_)

print("\nBest Cross Validation F1 Score:")
print(grid.best_score_)

# Best Model
best_model = grid.best_estimator_

# Predictions
y_pred = best_model.predict(X_test)

# Evaluation
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Grid Search Results
results = pd.DataFrame(grid.cv_results_)

results = results[
    [
        "params",
        "mean_train_score",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values("rank_test_score")


Fitting 5 folds for each of 10 candidates, totalling 50 fits
Best parameters:
{'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 10}

Best Cross Validation F1 Score:
0.5719514589128141

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.91      0.87      1035
           1       0.67      0.53      0.59       374

    accuracy                           0.81      1409
   macro avg       0.76      0.72      0.73      1409
weighted avg       0.80      0.81      0.80      1409



In [ ]:
df.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')

In [ ]:
df_encoded.columns

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn',
       'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes',
       'MultipleLines_No phone service', 'MultipleLines_Yes',
       'InternetService_Fiber optic', 'InternetService_No',
       'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
       'OnlineBackup_No internet service', 'OnlineBackup_Yes',
       'DeviceProtection_No internet service', 'DeviceProtection_Yes',
       'TechSupport_No internet service', 'TechSupport_Yes',
       'StreamingTV_No internet service', 'StreamingTV_Yes',
       'StreamingMovies_No internet service', 'StreamingMovies_Yes',
       'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check'],
      dtype='str')

In [32]:
df_encoded.columns.difference(df.columns)

Index(['Contract_One year', 'Contract_Two year', 'Dependents_Yes',
       'DeviceProtection_No internet service', 'DeviceProtection_Yes',
       'InternetService_Fiber optic', 'InternetService_No',
       'MultipleLines_No phone service', 'MultipleLines_Yes',
       'OnlineBackup_No internet service', 'OnlineBackup_Yes',
       'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
       'PaperlessBilling_Yes', 'Partner_Yes',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check',
       'PhoneService_Yes', 'StreamingMovies_No internet service',
       'StreamingMovies_Yes', 'StreamingTV_No internet service',
       'StreamingTV_Yes', 'TechSupport_No internet service', 'TechSupport_Yes',
       'gender_Male'],
      dtype='str')

In [59]:
service_cols = [
    'OnlineSecurity', 
    'OnlineBackup', 
    'DeviceProtection', 
    'TechSupport', 
    'StreamingTV', 
    'StreamingMovies', 
    'MultipleLines'
]

df['Total_Optional_Services'] = (df[service_cols] == 'Yes').sum(axis=1)

import numpy as np

# Use np.where to safely avoid dividing by zero!
df['Monthly_cost_per_service'] = np.where(
    df['Total_Optional_Services'] == 0, 
    df['MonthlyCharges'],  # If 0 services, just keep the standard monthly charge
    df['MonthlyCharges'] / df['Total_Optional_Services'] # Otherwise, do the division
)


In [60]:
# Create a mapping dictionary where higher numbers = higher risk
contract_mapping = {
    'Two year': 1,      # Low Risk
    'One year': 2,      # Medium Risk
    'Month-to-month': 3 # High Risk
}

# Apply the mapping to create the new column
df['Contract_Risk_Score'] = df['Contract'].map(contract_mapping)

In [61]:
# Check if the payment method contains 'Electronic check'
# We use .astype(int) to convert True to 1 and False to 0
df['Payment_Risk_Flag'] = (df['PaymentMethod'] == 'Electronic check').astype(int)

In [62]:
# Convert TotalCharges to numeric just in case there are blank strings ' '
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Find the 75th percentile (top 25% threshold) of TotalCharges
top_25_threshold = df['TotalCharges'].quantile(0.75)

# Flag customers who spend more than the top 25% threshold
df['High_Value_Customer'] = (df['TotalCharges'] >= top_25_threshold).astype(int)

In [63]:
# Define the edges of the bins (in months)
# Bins: 0-12, 12-24, 24-48, 48-100 (or max tenure)
bins = [0, 12, 24, 48, 100]

# Define the labels for each bin
labels = ['New', 'Growing', 'Established', 'Loyal']

# Cut the 'tenure' column into the specified bins
df['Customer_Lifecycle'] = pd.cut(df['tenure'], bins=bins, labels=labels, right=True)

# Note: Because this creates text labels, you will still need to use 
# pd.get_dummies() on this specific column later before feeding it to the model!

In [64]:
df.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn', 'Total_optional_service',
       'Contract_Risk_Score', 'Payment_Risk_Flag', 'High_Value_Customer',
       'Customer_Lifecycle', 'Monthly_cost_per_service',
       'Total_Optional_Services'],
      dtype='str')

In [ ]:
df_encoded = pd.get_dummies(df, drop_first=True)

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import mutual_info_classif

# 1. Create a temporary copy of your original dataframe so we don't mess up your main one
df_mi = df.copy()

# 2. We don't want to test the customer ID, so drop it if it's there
if 'customerID' in df_mi.columns:
    df_mi = df_mi.drop('customerID', axis=1)

# 3. Temporarily convert all text columns into simple numbers (0, 1, 2, etc.)
label_encoder = LabelEncoder()
for col in df_mi.select_dtypes(include=['object', 'category']).columns:
    df_mi[col] = label_encoder.fit_transform(df_mi[col].astype(str))

# 4. Separate X and y
y_mi = df_mi['Churn']
X_mi = df_mi.drop('Churn', axis=1)

# 5. Calculate Mutual Information Scores
mi_scores = mutual_info_classif(X_mi, y_mi, random_state=42)

# 6. View the clean results for your original columns!
mi_df = pd.DataFrame({'Feature': X_mi.columns, 'MI_Score': mi_scores})
print(mi_df.sort_values(by='MI_Score', ascending=False))

C:\Users\abhin\AppData\Local\Temp\ipykernel_24176\1176848437.py:15: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_mi.select_dtypes(include=['object', 'category']).columns:


                     Feature      MI_Score
6              MultipleLines  1.703819e-03
2                    Partner  1.419849e-03
13           StreamingMovies  1.419849e-03
15          PaperlessBilling  1.277865e-03
5               PhoneService  1.064887e-03
10          DeviceProtection  9.938946e-04
12               StreamingTV  9.229022e-04
0                     gender  9.229022e-04
20       Contract_Risk_Score  9.229022e-04
9               OnlineBackup  7.099247e-04
21         Payment_Risk_Flag  7.099247e-04
7            InternetService  4.969473e-04
23        Customer_Lifecycle  4.969473e-04
3                 Dependents  3.549624e-04
16             PaymentMethod  3.549624e-04
11               TechSupport  2.839699e-04
25   Total_Optional_Services  2.839699e-04
14                  Contract  2.839699e-04
1              SeniorCitizen  2.129774e-04
8             OnlineSecurity  2.129774e-04
22       High_Value_Customer  1.419849e-04
4                     tenure  7.099247e-05
19    Total

In [48]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score

X = df_encoded.drop("Churn", axis=1)
y = df_encoded["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [66]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline, make_pipeline

pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state = 42, max_iter=1000)
)

param_grid = {
    "logisticregression__C" : [0.01, 0.1, 1, 10],
    "logisticregression__penalty" : ['l1','l2'],
    "logisticregression__solver" :['liblinear', 'saga']
}

cv = StratifiedKFold(
    n_splits = 5,
    shuffle=True,
    random_state= 42
)

grid = GridSearchCV(
    estimator = pipeline,
    param_grid= param_grid,
    scoring='f1',
    n_jobs=-1,
    return_train_score=True,
    verbose=True,
    cv = cv
)

grid.fit(X_train, y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


ValueError: 
All the 80 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
40 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\abhin\Dropbox\PC\Downloads\projects\Customer Churn Prediction Platform\venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 851, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\abhin\Dropbox\PC\Downloads\projects\Customer Churn Prediction Platform\venv\Lib\site-packages\sklearn\base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\abhin\Dropbox\PC\Downloads\projects\Customer Churn Prediction Platform\venv\Lib\site-packages\sklearn\pipeline.py", line 649, in fit
    self._final_

In [ ]:
print("Best parameters:")
print(grid.best_params_)

print("Best F1-score:")
print(grid.best_score_)

best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

print("\n Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import pandas as pd

rf = RandomForestClassifier(random_state=42)

param_grid = {
    "n_estimators" : [100, 200, 300],
    "max_depth" : [None, 10, 20, 30],
    "min_samples_split" : [2, 5, 10],
    "min_samples_leaf" : [1, 2, 4],
    "max_features" : ["sqrt", "log2"]
}

cv = StratifiedKFold(
    n_splits = 5,
    shuffle= True,
    random_state=42
)

grid = GridSearchCV(
    estimator = rf,
    param_grid = param_grid,
    scoring = 'f1',
    cv= cv,
    n_jobs = -1,
    verbose=True,
    return_train_score=True
)

grid.fit(X_train, y_train)

print("Best parameters:")
print(grid.best_params_)

print("\nBest Cross Validation F1 Score:")
print(grid.best_score_)

# Best Model
best_model = grid.best_estimator_

# Predictions
y_pred = best_model.predict(X_test)

# Evaluation
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Grid Search Results
results = pd.DataFrame(grid.cv_results_)

results = results[
    [
        "params",
        "mean_train_score",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values("rank_test_score")


Fitting 5 folds for each of 216 candidates, totalling 1080 fits
